# Train a tiny GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/tiny-gpt.ipynb)

Train a small, modern transformer to write Shakespeare-like text, one character at a time. Companion to [Train a tiny GPT](../05-training/tiny-gpt.md).

**Hardware:** runs on CPU (several minutes), much faster on a GPU. In Colab: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import math, os, urllib.request
import torch
import matplotlib.pyplot as plt
from s2a.transformer import TransformerConfig
from s2a.gpt import GPT, train, estimate_loss, generate

device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("S2A_STEPS", 2000))   # total training steps
CONTEXT, BATCH = 64, 32
torch.manual_seed(0)
print("device:", device)

## 1. Data

About 1 MB of Shakespeare (the "tiny Shakespeare" file from Andrej Karpathy's char-rnn). Each character is a token, so there's no tokenizer to train. The last 10% is held out to check the model isn't just memorizing.

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    text = urllib.request.urlopen(URL, timeout=30).read().decode("utf-8")
except Exception as e:
    print("Download failed, using a small built-in text instead:", e)
    text = "To be, or not to be, that is the question.\nAll the world's a stage, and all the men and women merely players.\n" * 200

chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(chars[i] for i in ids)

data = torch.tensor(encode(text))
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"{len(text):,} characters, vocabulary of {len(chars)}")
print(text[:200])

## 2. The model

The same architecture as the [transformer block lab](transformer-block.ipynb): 4 blocks, RMSNorm, grouped-query attention with RoPE, SwiGLU.

In [ ]:
cfg = TransformerConfig(vocab_size=len(chars), d_model=128, n_layers=4, n_heads=4, n_kv_heads=2, d_ff=344)
model = GPT(cfg).to(device)
print(f"{model.num_params():,} parameters")

## 3. Before training

An untrained model guesses almost uniformly, so its loss is close to `ln(vocabulary size)` and its output is noise.

In [ ]:
print(f"loss {estimate_loss(model, val_data, context=CONTEXT):.3f} vs ln({len(chars)}) = {math.log(len(chars)):.3f}")
print(decode(generate(model, encode("\n"), 200, context=CONTEXT)))

## 4. Train

Each step: take a random batch of text windows, predict every next character, measure the loss, compute gradients, and nudge the weights. We check the held-out loss ten times along the way.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
losses, val_points = [], []
chunk = max(1, STEPS // 10)
for i in range(10):
    losses += train(model, train_data, steps=chunk, batch_size=BATCH, context=CONTEXT, optimizer=optimizer)
    val = estimate_loss(model, val_data, context=CONTEXT)
    val_points.append((len(losses), val))
    print(f"step {len(losses):>5}: train loss {sum(losses[-chunk:]) / chunk:.3f}, held-out loss {val:.3f}")

In [ ]:
plt.plot(losses, alpha=0.4, label="train (each step)")
plt.plot(*zip(*val_points), "o-", label="held-out")
plt.axhline(math.log(len(chars)), ls="--", c="gray", label="uniform guessing")
plt.xlabel("step"); plt.ylabel("loss (nats per character)"); plt.legend(); plt.show()

## 5. After training

Same sampling code as before. Look for spelling, line breaks and speaker names: all learned from predicting the next character.

In [ ]:
for t in [0.5, 1.0]:
    print(f"--- temperature {t} ---")
    print(decode(generate(model, encode("\n"), 400, context=CONTEXT, temperature=t)))

## Try it

- Train for 5,000 steps (`STEPS = 5000` at the top). Does the held-out loss keep falling, or pull away from the train loss?
- Double `d_model` and `d_ff`, or the number of layers. What changes in parameter count, speed and loss?
- Change the learning rate to `1e-2` and to `1e-4`. What do the loss curves look like?
- Set `n_layers=1`. Which abilities does the model lose first?
- `generate` recomputes the whole context for every new character. How would a [KV cache](../06-inference/kv-cache.md) avoid that?